# NB 2.2 &mdash; Solucions dels exercicis

**MP 5134** &mdash; UT2 · *Dades: AEMET, estació de l'aeroport de Palma*

---

Solucionari dels cinc exercicis de la secció 9 del
[NB 2.2](NB_2_2_regressio_multiple.ipynb) i orientacions per al debat de la
secció 10.

La primera cel·la reconstrueix l'estat del notebook original, **amb el test
segellat inclòs**: també als solucionaris es respecta el segell.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, r2_score

URL_DADES = "https://raw.githubusercontent.com/pprohenspolitecnicllevant/disseny-avaluacio-models-ml/refs/heads/main/UT01-Entorn_de_treball_primer_model/aemet/meteo_palma.csv"
df = pd.read_csv(URL_DADES, parse_dates=["fecha"])

# El test segellat (2024-2025) no es toca fins a la UT10
TALL_SEGELLAT = "2024-01-01"
dades = df[df["fecha"] < TALL_SEGELLAT].copy()

FEATURES = ["tmax", "tmin", "tmed", "sol", "presMax", "presMin",
            "prec", "velmedia", "mes", "dia_any"]
OBJECTIU = "tmax_dema"
dades = dades.dropna(subset=FEATURES + [OBJECTIU])

TALL_VALIDACIO = "2022-01-01"
train = dades[dades["fecha"] < TALL_VALIDACIO]
valid = dades[dades["fecha"] >= TALL_VALIDACIO].copy()

model_1 = LinearRegression().fit(train[["tmax"]], train[OBJECTIU])
model_tot = LinearRegression().fit(train[FEATURES], train[OBJECTIU])
print("Estat reconstruït.")

## Exercici 1

> Entrena un model només amb `sol`. Afegeix-hi `tmax`. Dibuixa el núvol de `sol`
> contra `tmax_dema`.

In [ ]:
for variables in (["sol"], ["sol", "tmax"]):
    m = LinearRegression().fit(train[variables], train[OBJECTIU])
    pred = m.predict(valid[variables])
    print(f"{str(variables):>16}: MAE {mean_absolute_error(valid[OBJECTIU], pred):.2f} °C   R2 {r2_score(valid[OBJECTIU], pred):.2f}")

plt.figure(figsize=(7, 4))
plt.scatter(train["sol"], train[OBJECTIU], alpha=0.15, s=8)
plt.xlabel("Hores de sol d'avui")
plt.ylabel("Màxima de demà (°C)")
plt.show()

**Resposta.** Només amb el sol, el model s'equivoca uns **4,7 °C** i té un R2 de
0,29. En afegir-hi `tmax`, l'error cau a 1,61 °C, **exactament el mateix que amb
`tmax` sola**: el sol no aporta res un cop coneixem la temperatura.

El núvol explica per què. Els dies de moltes hores de sol tendeixen a ser més
calorosos, sí, però a cada nombre d'hores de sol hi ha dies de 15 °C i dies de
30 °C. El motiu principal és l'època de l'any: **un dia serè de gener té unes 8 o 9
hores de sol i fa fred; un dia serè de juliol en té 13 i fa calor**. Les hores de
sol depenen molt de la llargada del dia, i la temperatura, encara més.

*Per a la correcció:* la idea que cal que surti és que *una variable pot estar
relacionada amb l'objectiu i alhora no afegir res si ja tenim una altra variable
que conté la mateixa informació*. És el mateix que passava amb `tmed` a la
taula de la secció 6.

## Exercici 2

> Amb `model_1`, calcula el MAE de validació per a cada mes.

In [ ]:
valid["error_abs"] = (model_1.predict(valid[["tmax"]]) - valid[OBJECTIU]).abs()

per_mes = valid.groupby("mes")["error_abs"].mean().round(2)

plt.figure(figsize=(8, 3))
per_mes.plot(kind="bar")
plt.ylabel("MAE (°C)")
plt.xlabel("Mes")
plt.xticks(rotation=0)
plt.show()

per_mes

**Resposta.** Els mesos més difícils són el **febrer, l'abril i el maig**, amb
errors al voltant de 2 °C. Els més fàcils són l'**octubre** i el **desembre**, per
sota d'1,4 °C, i els mesos d'estiu també surten bé.

Una hipòtesi raonable: a la primavera el temps és més canviant, amb entrades
d'aire fred i dies de calor que s'alternen, i *demà igual que avui* falla més. A
l'estiu, a Mallorca, un dia calorós sol anar seguit d'un altre dia calorós.

*Per a la correcció:* no s'avalua que la hipòtesi meteorològica sigui correcta,
sinó que l'alumnat **faci servir el desglossament per explicar on falla el
model**. És una tècnica que faran servir tot el curs: una mitjana global amaga
diferències grans entre grups.

## Exercici 3

> Calcula la desviació percentual del `model_1` sobre la validació, en graus
> Celsius i en kelvins.

In [ ]:
real = valid[OBJECTIU]
predit = model_1.predict(valid[["tmax"]])

print(f"En °C:  MAE {mean_absolute_error(real, predit):.2f}   desviació {100 * mean_absolute_percentage_error(real, predit):.1f} %")
print(f"En K:   MAE {mean_absolute_error(real + 273.15, predit + 273.15):.2f}   desviació {100 * mean_absolute_percentage_error(real + 273.15, predit + 273.15):.2f} %")

**Resposta.** El MAE no canvia gens: 1,61 °C i 1,61 K són la mateixa diferència de
temperatura. La desviació percentual, en canvi, passa del **7,5% al 0,5%**, i el
model és exactament el mateix.

El motiu és que el percentatge divideix per la temperatura real, i **el zero de
les escales de temperatura és arbitrari**: el 0 °C és on es glaça l'aigua, i el
0 K és el zero absolut. Segons on posis el zero, el mateix error sembla gran o
petit.

La conclusió pràctica: **la desviació percentual només té sentit quan el zero de la
variable vol dir *res***: zero grams, zero euros, zero unitats venudes. Per a
temperatures, no s'ha de fer servir. A Palma no hi ha gaires dies amb màximes
a prop de 0 °C, però en una estació del Pirineu el percentatge sortiria
directament absurd.

*Per a la correcció:* és un exercici curt però amb una lliçó molt valuosa sobre
triar mètriques. Connecta directament amb el debat de les vendes del NB 2.1.

## Exercici 4

> Amb el model de totes les variables, calcula el MAE per separat per al 2022 i
> per al 2023.

In [ ]:
valid["error_tot"] = (model_tot.predict(valid[FEATURES]) - valid[OBJECTIU]).abs()

valid.groupby(valid["fecha"].dt.year)[["error_tot", "error_abs"]].mean().round(3).rename(
    columns={"error_tot": "MAE deu variables", "error_abs": "MAE només tmax"}
)

**Resposta.** El 2022 surt un MAE d'uns **1,59 °C** i el 2023, d'uns **1,49 °C**.
Hi ha una desena de grau de diferència entre un any i l'altre, amb el mateix model.

I aquí hi ha la lliçó de fons. Al NB 2.2 vam celebrar que passar d'una variable a
deu millorava l'error **set centèsimes**. Ara veiem que **només canviant l'any de
validació l'error es mou una desena**, més que tota la millora. Amb una sola xifra
de validació, no podem estar segurs que les set centèsimes siguin reals i no
sort.

*Per a la correcció:* és exactament el problema que resol la validació creuada de la
**UT7**, i convé dir-ho explícitament. Qui llegeixi també la columna de `tmax`
sola veurà que la diferència entre els dos models es manté tots dos anys, que és
un indici (no una prova) que la millora és real.

## Exercici 5

> Explica per què no s'ha de calcular el MAE sobre el 2024 i el 2025 "només per
> mirar".

**Resposta orientativa:**

> Encara que no canviem res conscientment, un cop hem vist el resultat sobre el test
> ja no el podem oblidar, i totes les decisions que prenguem després (quines
> variables fem servir, quin model triem) quedaran influïdes per aquell número.
> El test deixaria de ser dades *noves* per al nostre procés de treball. A la UT10
> ja no tindríem cap manera honesta de saber com funciona el model amb dies que
> no hem vist mai, perquè l'únic tros reservat per a això ja l'hauríem gastat.

*Per a la correcció:* la paraula clau és **decisions**. Una resposta que només digui
*"perquè ho diu el professor"* o *"perquè és trampa"* és insuficient. Cal que
aparegui la idea que mirar el test **contamina les decisions posteriors**, encara
que no toquem el codi.

## Orientacions per al debat

**Quin model posaries en marxa?** La resposta més defensable és el d'una o dues
variables. Set centèsimes de grau no compensen haver de tenir deu mesures
disponibles cada dia, i l'exercici 4 demostra que ni tan sols estem segurs que la
millora sigui real. Però si algú defensa el model gran perquè les mesures ja
arriben totes juntes de l'estació automàtica, també és un argument vàlid.

**On podrien valer diners unes centèsimes?** En empreses que operen a gran escala:
la demanda elèctrica d'una regió depèn molt de la temperatura (aire condicionat),
i un error petit multiplicat per milions de llars és molta energia. En turisme o en
agricultura, en canvi, un grau amunt o avall no canvia cap decisió.

**Serveix l'aprenentatge automàtic per predir la temperatura?** Sí, però **no amb
les dades d'una sola estació d'ahir**. Els serveis meteorològics fan servir
dades de milers d'estacions, satèl·lits i models físics de l'atmosfera. El nostre
model no sap que ve un front fred des del nord perquè no li hem donat aquesta
informació. És un bon moment per recordar que **el límit d'un model el posen
sovint les dades, no l'algorisme**.